# Tutorial 5 - Conversion

While CG simulations are fantastic at studying the longer timescale properties, such as lipid organisation and binding, we might want to study some of this further in atomistic detail. This could be converting the whole system to AT resolution to simulate or taking parts of the system to seed new types of simulations (for example, a ligand bound).

Luckily for us, there are tools to backmap Martini CG resolution to an AT resolution. One of the original tools was _backward_ (for which there is a tutorial [here](https://cgmartini.nl/docs/tutorials/Martini3/Backward/) for you to look at in your own time) and there have been others published since. 

One of these, which we will use today, is CG2AT (original publication [here](https://pubs.acs.org/jctcce/article/17/10/6472/873655/CG2AT2-an-Enhanced-Fragment-Based-Approach-for)). This uses a fragment-based approach to reconstruct each molecule in the system into AT resolution. There is a slightly more comprehensive tutorial for you to look at [here](https://cgmartini.nl/docs/tutorials/Martini3/cg2at/), but we will convert the system we have looked at today here below. 

First, let's look at a slightly reduced version of cg2at and what the input options are:

In [ ]:
!cg2at_lite -h

In [ ]:
!cp ../tutorial_4/md.pdb ../tutorial_4/md.tpr .
!cp ../tutorial_1/VDAC1_clean.pdb .

In [ ]:
%%bash

## For the sake of time, we will only include the protein and lipids in our conversion

gmx make_ndx -f md.pdb -o sys.ndx << EOF
rPOPC|rPOPE|rPOPI
name 18 LIPID
1|18
q
EOF

gmx trjconv -f md.pdb -s md.tpr -o protein_lipid.pdb -n sys.ndx << EOF
19
EOF

There are a few options for the input here, but we will focus on the ones used in this tutorial:

- `-c` which specifies the input file for your CG system. This will be a .gro or .pdb file
- `-a` which gives the atomistic coordinates for your protein which was simulated so the programme can use this to align to. This is optional. This will be a .gro or a .pdb file
- `-fg` the name of the CG force field that was used for the input system. Since Martini has recently had a large lipidome update, the one we will use here is `martini_3-1_new_lipidome_charmm36`
- `-loc` the location of where to save the output data, the `.` just means the directory we are working in

In [ ]:
## This might take a little while to run as there are small equilibration steps included
!cg2at_lite -c protein_lipid.pdb -a VDAC1_clean.pdb -fg martini_3-1_new_lipidome_charmm36 -loc .

We can look at this structure!

In [ ]:
from IPython.display import HTML, display
import base64, os


def view_atomistic(
    structure_file: str,
    height: int = 600,
    show_cartoon: bool = True,        # Ribbon/cartoon for secondary structure
    show_ball_and_stick: bool = True, # Bonds + atoms for all heavy atoms
    show_spacefill: bool = False,     # VdW spheres (CPU-heavy for large systems)
    cartoon_alpha: float = 0.85,      # Cartoon opacity
    stick_size: float = 0.16,         # Bond stick radius in Å
    ball_size: float = 0.25,          # Atom sphere radius factor (ball-and-stick)
    spacefill_alpha: float = 0.25,    # VdW sphere opacity (if enabled)
) -> HTML:
    """
    Visualise an atomistic protein structure in Mol* inside Jupyter.

    Representations
    ---------------
    cartoon       → ribbon coloured by secondary structure (helix/sheet/coil)
    ball-and-stick→ heavy atom spheres + covalent bonds as sticks
    spacefill     → full VdW spheres (optional; disabled by default for speed)

    Parameters
    ----------
    structure_file : str
        Path to PDB, mmCIF/CIF, or GRO file.

    height : int
        Viewer height in pixels.

    show_cartoon : bool
        Render cartoon/ribbon secondary-structure representation.

    show_ball_and_stick : bool
        Render bonds and atoms as balls and sticks.

    show_spacefill : bool
        Overlay semi-transparent VdW spheres.  Good for surface visualisation
        but expensive for large systems (>50 k atoms).

    cartoon_alpha : float
        Cartoon opacity (0–1).

    stick_size : float
        Bond stick radius in Å.

    ball_size : float
        Atom sphere scale factor for ball-and-stick (relative to element radius).

    spacefill_alpha : float
        Opacity of VdW spheres (0–1).  Only used when show_spacefill=True.

    Returns
    -------
    IPython.display.HTML
        Call display() on the return value, or let Jupyter auto-display it.
    """
    # ── Load and encode structure file ────────────────────────────────────────
    with open(structure_file, "rb") as fh:
        b64 = base64.b64encode(fh.read()).decode()

    ext = os.path.splitext(structure_file)[1].lstrip(".").lower()
    fmt = {"gro": "gro", "pdb": "pdb", "cif": "mmcif", "mmcif": "mmcif"}.get(ext, "pdb")

    # Convert Python bools to JS booleans
    js_cartoon        = str(show_cartoon).lower()
    js_ball_and_stick = str(show_ball_and_stick).lower()
    js_spacefill      = str(show_spacefill).lower()

    page = f"""<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8"/>
  <link rel="stylesheet"
        href="https://cdn.jsdelivr.net/npm/molstar@latest/build/viewer/molstar.css"/>
  <style>
    html, body {{ margin:0; padding:0; height:100%; background:#1a1a2e; }}
    #app {{ position:absolute; inset:0; }}
  </style>
</head>
<body>
  <div id="app"></div>
  <script src="https://cdn.jsdelivr.net/npm/molstar@latest/build/viewer/molstar.js"></script>
  <script>
  (async () => {{

    // ── 1. Create Mol* viewer ────────────────────────────────────────────────
    const viewer = await molstar.Viewer.create('app', {{
      layoutIsExpanded:       false,
      layoutShowControls:     false,
      layoutShowLeftPanel:    false,
      layoutShowSequence:     false,   // sequence bar useful for atomistic work
      layoutShowLog:          false,
      viewportShowAnimation:  false,
      viewportShowExpand:     true,
    }});
    const plugin = viewer.plugin;

    // ── 2. Decode base64 structure ───────────────────────────────────────────
    const structText = atob("{b64}");

    // ── 3. Build the structure ───────────────────────────────────────────────
    const rawData   = await plugin.builders.data.rawData(
      {{ data: structText }},
      {{ state: {{ isGhost: true }} }}
    );
    const trajectory = await plugin.builders.structure.parseTrajectory(rawData, '{fmt}');
    const model      = await plugin.builders.structure.createModel(trajectory);
    const structure  = await plugin.builders.structure.createStructure(model);

    const colorTheme = {{ name: 'chain-id' }};

    // ── 4. Cartoon / ribbon ──────────────────────────────────────────────────
    if ({js_cartoon}) {{
      await plugin.builders.structure.representation.addRepresentation(structure, {{
        type: 'cartoon',
        colorTheme: colorTheme,
        typeParams: {{
          alpha: {cartoon_alpha},
        }},
      }});
    }}

    // ── 5. Ball-and-stick ────────────────────────────────────────────────────
    if ({js_ball_and_stick}) {{
      await plugin.builders.structure.representation.addRepresentation(structure, {{
        type: 'ball-and-stick',
        colorTheme: colorTheme,
        typeParams: {{
          sizeFactor:     {stick_size},
          ballSizeFactor: {ball_size},
          bondSpacing:    1.0,
        }},
      }});
    }}

    // ── 6. VdW spacefill (optional) ──────────────────────────────────────────
    if ({js_spacefill}) {{
      await plugin.builders.structure.representation.addRepresentation(structure, {{
        type: 'spacefill',
        colorTheme: colorTheme,
        sizeTheme:  {{ name: 'physical' }},   // true VdW radii per element
        typeParams: {{
          alpha: {spacefill_alpha},
        }},
      }});
    }}

    // ── 7. Fit camera ────────────────────────────────────────────────────────
    plugin.canvas3d?.requestCameraReset();

  }})();
  </script>
</body>
</html>"""

    escaped = page.replace("'", "&#39;")
    return HTML(
        f'<iframe srcdoc=\'{escaped}\' '
        f'style="width:100%;height:{height}px;border:none;border-radius:6px;"></iframe>'
    )


### Below is the lines that actually executes the visulization

display(view_atomistic("FINAL/final_cg2at_aligned.pdb", show_ball_and_stick=True))


## That brings us to the end of Martini CG tutorials!!

If anything is unclear, please ask! Similarly, if you are wondering how to apply anything here to your own systems, this is the perfect time to discuss it. In addition to these tutorials, there are many others on the [Martini website](https://cgmartini.nl/docs/tutorials/Martini3/tutorials.html) for you to look through and try out in your own time (software needed might not be installed on the cloud here). 